# 📊 Data Exploration and Visualization

This notebook explores the dataset before training the SLM model.

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from tqdm.auto import tqdm

from src.data import Tokenizer, DatasetProcessor

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Dataset

In [ ]:
# Load dataset
dataset = load_dataset("tiny_shakespeare", split="train")
print(f"Dataset size: {len(dataset)} samples")

# Create train/val split
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_dataset['train']
val_data = split_dataset['test']

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

In [ ]:
# Display sample texts
print("Sample texts:")
print("="*50)
for i in range(3):
    print(f"Sample {i+1}:")
    print(train_data[i]['text'][:200] + "...")
    print("-"*50)

## 2. Text Statistics

In [ ]:
# Calculate text lengths
text_lengths = [len(text['text']) for text in train_data]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(text_lengths, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Text Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Text Lengths')
axes[0].axvline(np.mean(text_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(text_lengths):.0f}')
axes[0].axvline(np.median(text_lengths), color='green', linestyle='--', label=f'Median: {np.median(text_lengths):.0f}')
axes[0].legend()

# Box plot
axes[1].boxplot(text_lengths, vert=True)
axes[1].set_ylabel('Text Length (characters)')
axes[1].set_title('Text Length Distribution')
axes[1].set_xticklabels(['All Texts'])

plt.tight_layout()
plt.show()

print(f"Mean length: {np.mean(text_lengths):.2f} characters")
print(f"Median length: {np.median(text_lengths):.2f} characters")
print(f"Max length: {np.max(text_lengths)} characters")
print(f"Min length: {np.min(text_lengths)} characters")
print(f"Std deviation: {np.std(text_lengths):.2f} characters")

## 3. Tokenization Analysis

In [ ]:
# Initialize tokenizer
tokenizer = Tokenizer()
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Tokenize sample texts
sample_text = train_data[0]['text']
tokens = tokenizer.encode(sample_text)

print(f"\nSample text: {sample_text[:100]}...")
print(f"Number of tokens: {len(tokens)}")
print(f"First 20 tokens: {tokens[:20]}")
print(f"Decoded tokens: {tokenizer.decode(tokens[:20])}")

In [ ]:
# Token length distribution
token_lengths = []
for text in tqdm(train_data, desc="Tokenizing"):
    tokens = tokenizer.encode(text['text'])
    token_lengths.append(len(tokens))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(token_lengths, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Tokens')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Token Counts')
axes[0].axvline(np.mean(token_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(token_lengths):.0f}')
axes[0].axvline(np.median(token_lengths), color='green', linestyle='--', label=f'Median: {np.median(token_lengths):.0f}')
axes[0].legend()

# Box plot
axes[1].boxplot(token_lengths, vert=True)
axes[1].set_ylabel('Number of Tokens')
axes[1].set_title('Token Count Distribution')
axes[1].set_xticklabels(['All Texts'])

plt.tight_layout()
plt.show()

print(f"Mean tokens: {np.mean(token_lengths):.2f}")
print(f"Median tokens: {np.median(token_lengths):.2f}")
print(f"Max tokens: {np.max(token_lengths)}")
print(f"Min tokens: {np.min(token_lengths)}")

## 4. Vocabulary Analysis

In [ ]:
# Analyze vocabulary
all_tokens = []
for text in tqdm(train_data, desc="Collecting tokens"):
    tokens = tokenizer.encode(text['text'])
    all_tokens.extend(tokens)

token_counts = Counter(all_tokens)
unique_tokens = len(token_counts)

print(f"Total tokens: {len(all_tokens):,}")
print(f"Unique tokens: {unique_tokens:,}")
print(f"Vocabulary coverage: {unique_tokens / tokenizer.vocab_size * 100:.2f}%")

In [ ]:
# Most common tokens
most_common = token_counts.most_common(20)

tokens, counts = zip(*most_common)
token_texts = [tokenizer.decode([t]) for t in tokens]

plt.figure(figsize=(12, 6))
plt.barh(range(len(tokens)), counts)
plt.yticks(range(len(tokens)), token_texts)
plt.xlabel('Frequency')
plt.title('Most Common Tokens')
plt.tight_layout()
plt.show()

## 5. Dataset Statistics Summary

In [ ]:
# Create summary DataFrame
summary_data = {
    'Metric': [
        'Total Samples',
        'Total Tokens',
        'Unique Tokens',
        'Avg Tokens/Sample',
        'Median Tokens/Sample',
        'Max Tokens/Sample',
        'Min Tokens/Sample',
        'Vocabulary Size',
        'Vocab Coverage (%)'
    ],
    'Value': [
        len(train_data),
        f"{len(all_tokens):,}",
        f"{unique_tokens:,}",
        f"{np.mean(token_lengths):.2f}",
        f"{np.median(token_lengths):.2f}",
        np.max(token_lengths),
        np.min(token_lengths),
        tokenizer.vocab_size,
        f"{unique_tokens / tokenizer.vocab_size * 100:.2f}%"
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df

## 6. Save Statistics

Save the statistics for future reference.

In [ ]:
# Save statistics
summary_df.to_csv('../data/dataset_statistics.csv', index=False)
print("Statistics saved to data/dataset_statistics.csv")

# Save token distribution
np.save('../data/token_lengths.npy', np.array(token_lengths))
print("Token lengths saved to data/token_lengths.npy")